# Reproduce QM9 Semi-Supervised Results

This notebook reproduces the final selected experiments showcase in the report. It simply calls `src/run.py` with the final Hydra configs.

Showcased experiments:
- Supervised main split
- Mean Teacher main split
- Peer consistency main split
- Supervised low-label split
- Mean Teacher low-label split
- Peer consistency low-label split
- Augmentation ablation runs


## 1. Environment Setup

The notebook assumes the repository is already cloned and that you are in the repo root. If not, clone it first and `cd` into it.

In [ ]:
# If needed, create the environment with uv and install dependencies
!uv sync

# If your environment already exists, only verify the key packages:
import sys
print(sys.executable)

## 2. W&B Login

The project logs all training runs to Weights & Biases. Before reproducing runs, log in once in this notebook session.
If you do not want to log to W&B, set `logger.disable=true` in the commands below.


In [ ]:
import wandb
wandb.login()

## 3. Reproduce the experiments from report
The commands are written so you can run them sequentially. If you want to run on cpu (will take very long time) remove the `device=cuda` from command. When not using wandb pass `logger.disable=true`. You can also customize number of workers depending of amount of available cpu cores through `dataset.init.num_workers`

### 3.1 Main split runs

In [ ]:
# Supervised control on the main split
!python src/run.py device=cuda dataset.init.num_workers=4 trainer.init.consistency_weight=0 logger.name=supervised_control_main_split


In [ ]:
# Mean Teacher on the main split
!python src/run.py device=cuda dataset.init.num_workers=4 trainer.init.ssl_method=mt trainer.init.mt_augment_mode=noise trainer.init.mt_augment_strength=0.1 trainer.init.consistency_weight=1 logger.name=mt_main_split_noise_s01


In [ ]:
# Peer consistency on the main split
!python src/run.py device=cuda dataset.init.num_workers=4 trainer.init.ssl_method=ncps trainer.init.num_models=2 trainer.init.epoch_mode=labeled trainer.init.ncps_augment_mode=noise trainer.init.ncps_augment_strength=0.05 trainer.init.consistency_weight=0.1 logger.name=peer_main_split_noise_l01


### 3.2 Low-label split runs

In [ ]:
# Supervised low-label control (3% labeled)
!python src/run.py device=cuda dataset.init.num_workers=4 dataset.init.splits='[0.9,0.03,0.035,0.035]' trainer.init.consistency_weight=0 trainer.train.total_epochs=500 logger.name=supervised_lowlabel_3pct


In [ ]:
# Mean Teacher low-label (3% labeled, noise)
!python src/run.py device=cuda seed=0 dataset.init.num_workers=4 dataset.init.splits='[0.9,0.03,0.035,0.035]' trainer.init.ssl_method=mt trainer.init.mt_augment_mode=noise trainer.init.mt_augment_strength=0.1 trainer.init.consistency_weight=1 logger.name=mt_lowlabel_noise_s01


In [ ]:
# Peer consistency low-label (3% labeled, noise)
!python src/run.py device=cuda seed=0 dataset.init.num_workers=4 dataset.init.splits='[0.9,0.03,0.035,0.035]' trainer.init.ssl_method=ncps trainer.init.num_models=2 trainer.init.epoch_mode=labeled trainer.init.ncps_augment_mode=noise trainer.init.ncps_augment_strength=0.05 trainer.init.consistency_weight=0.1 logger.name=peer_lowlabel_noise_l01


### 3.3 Augmentation ablation runs

In [ ]:
# Augmentation ablation 1: Mean Teacher with feature noise
!python src/run.py device=cuda seed=0 dataset.init.num_workers=4 trainer.init.ssl_method=mt trainer.init.mt_augment_mode=noise trainer.init.mt_augment_strength=0.1 trainer.init.consistency_weight=1 logger.name='mt_noise_s01_l1'


In [ ]:
# Augmentation ablation 2: Mean Teacher with feature masking
!python src/run.py device=cuda seed=0 dataset.init.num_workers=4 trainer.init.ssl_method=mt trainer.init.mt_augment_mode=mask trainer.init.mt_augment_strength=0.1 trainer.init.consistency_weight=1 logger.name='mt_mask_l1'


In [ ]:
# Augmentation ablation 3: peer consistency with edge dropout
!python src/run.py device=cuda seed=0 dataset.init.num_workers=4 trainer.init.ssl_method=ncps trainer.init.num_models=2 trainer.init.epoch_mode=labeled trainer.init.ncps_augment_mode=edge_dropout trainer.init.ncps_augment_strength=0.1 trainer.init.consistency_weight=1 logger.name='peer_edge_l1'


In [ ]:
# Augmentation ablation 4: peer consistency with graph augmentation
!python src/run.py device=cuda seed=0 dataset.init.num_workers=4 trainer.init.ssl_method=ncps trainer.init.num_models=2 trainer.init.epoch_mode=labeled trainer.init.ncps_augment_mode=graph trainer.init.ncps_augment_strength=0.1 trainer.init.consistency_weight=1 logger.name='peer_graph_l1'